#PARTIE 1 : Constitution d'une base nationale des avis Google Maps des pharmacies


Objectif
--------
Développer un pipeline automatisé permettant de constituer une base de données
des coordonnées maps permettant de receuillir les avis Google Maps de l'ensemble des pharmacies de France métropolitaine.

Pipeline général
----------------
1. Sélection des variables pertinentes.
2. Nettoyage et préparation des données.
3. Constitution de la base finale pour l'analyse.

Résultat
---------
Dans cette partie le pipeline produit une base de données structurée contenant les informations
des pharmacies et l'ensemble des régions associées ainsi que leurs coordonnées Google Maps associés, directement
exploitable pour le scraping, des analyses statistiques et des travaux de traitement
automatique du langage naturel.


Nos bases de référence ont été respectivement obtenue sur data.gouv relié au site de l'Institut nationale de l'information géographique et forestière, aussi OpenStreetMap, la Base Adresse Nationale (BAN) et l'INSEE

Chargement des données (bibliothèques et data frame)

In [ ]:
import pandas as pd
import geopandas as gpd
from shapely import wkt
import matplotlib.pyplot as plt

La base retenue a servi de point d'entrée à l'ensemble du pipeline de préparation des données puis au processus automatisé de collecte des avis.

Elles a été retenue selon les critères suivant :
* couverture nationale ( base provenant de openstreetmaps );
* identifiant unique pour chaque pharmacie ;
* coordonnées géographiques disponibles ;
* qualité et cohérence des données ;
* complétude des données

In [ ]:
df_pharma = pd.read_csv("/content/pharmacies_point.csv"),

/tmp/ipykernel_41747/2006319225.py:1: DtypeWarning: Columns (4,11,16,32,34) have mixed types. Specify dtype option on import or set low_memory=False.
  df_pharma = pd.read_csv("/content/pharmacies_point.csv")


In [ ]:
regions = gpd.read_file("/content/regions.geojson")

Conversion en géométrie

In [ ]:
df_pharma["geometry"] = df_pharma["the_geom"].apply(wkt.loads)

In [ ]:
gdf_pharma = gpd.GeoDataFrame(
    df_pharma,
    geometry="geometry"
)

In [ ]:
type(gdf_pharma)

geopandas.geodataframe.GeoDataFrame

Diagnostique spaciale ici on cherch à vérifier si nous avons les mêmes types de coordonnées géospatiales

In [ ]:
print(gdf_pharma.crs)
print(regions.crs)

None
EPSG:4326


In [ ]:
print(gdf_pharma.total_bounds)

[-567126.04027365 5069749.45167958 1062830.74203263 6633728.68620921]


In [ ]:
regions.geom_type.value_counts()

,count
Polygon,9
MultiPolygon,9


Identification du CRS

In [ ]:
#gdf_pharma = gdf_pharma.set_crs(
 #   "EPSG:2154"
#)

In [ ]:
gdf_pharma = gdf_pharma.set_crs("EPSG:3857", allow_override=True)
regions_3857 = regions.to_crs("EPSG:3857")

Harmonisation des crs

In [ ]:
#gdf_pharma = gdf_pharma.to_crs(regions.crs)

controle

In [ ]:
print(gdf_pharma.crs)
print(regions.crs)

EPSG:3857
EPSG:4326


Contôle visuel

Jointure spaciale

In [ ]:
result = gpd.sjoin(gdf_pharma, regions_3857, how="left", predicate="within")

In [ ]:
print(result.columns)

Index(['FID', 'osm_id', 'amenity', 'name', 'short_name', 'official_name',
       'alt_name', 'old_name', 'operator', 'operator-type', 'dispensing',
       'emergency', 'capacity', 'wheelchair', 'social_facility',
       'ref-FR-FINESS', 'type-FR-FINESS', 'ref-FR-NAF', 'ref-FR-SIRET',
       'website', 'contact-website', 'url', 'phone', 'contact-phone', 'fax',
       'contact-fax', 'email', 'contact-email', 'addr-housename',
       'addr-housenumber', 'addr-street', 'addr-city', 'addr-postcode',
       'wikidata', 'wikipedia', 'description', 'opening_hours', 'source',
       'note', 'osm_version', 'osm_timestamp', 'the_geom', 'osm_original_geom',
       'osm_type', 'geometry', 'index_right', 'code', 'nom'],
      dtype='object')


# Nous avons décidé de selectionner les variables suivantes comme variable pertinente de l'étude :


*  FID         :  Identifiant unique de la pharmacie
* addr-postcode : Code postal                     
* addr-city     : Nom de la ville             
* the_geom      : Coordonnées GPS (latitude/longitude)
* addr-street   : Adresse                              
* name    : Nom de la pharmacie  


In [ ]:
print(result[['FID', 'name', 'nom', 'addr-postcode']])

                                FID                         name  \
0       pharmacies_point.4534591198  Pharmacie Centrale de Bondy   
1        pharmacies_point.172156934       Pharmacie des Andaines   
2       pharmacies_point.3918927589   Pharmacie Cayeux Etaploise   
3       pharmacies_point.5101170481           Pharmacie Lorraine   
4      pharmacies_point.11429216164        Pharmacie de la Halle   
...                             ...                          ...   
19132   pharmacies_point.1008156074            Pharmacie Breuzin   
19133  pharmacies_point.13179599050   Pharmacie de la République   
19134   pharmacies_point.3712079795         Pharmacie des Arènes   
19135   pharmacies_point.1933342224            Pharmacie Jacques   
19136   pharmacies_point.1152714441               Pharmacie Mery   

                       nom addr-postcode  
0            Île-de-France         93140  
1                Normandie           NaN  
2          Hauts-de-France         62630  
3          

In [ ]:
result['FID'].isna().sum()

np.int64(0)

In [ ]:
result['nom'].isna().sum()

np.int64(70)

In [ ]:
result['nom'].unique()

array(['Île-de-France', 'Normandie', 'Hauts-de-France', 'Grand Est',
       'Auvergne-Rhône-Alpes', 'Pays de la Loire', 'Occitanie',
       'Centre-Val de Loire', "Provence-Alpes-Côte d'Azur",
       'Nouvelle-Aquitaine', 'Bourgogne-Franche-Comté', 'Bretagne', nan,
       'Corse'], dtype=object)

In [ ]:
result['name'].isna().sum()

np.int64(1265)

Contrôle de la qualité des jointures

In [ ]:
result["index_right"].isna().sum()

np.int64(70)

In [ ]:
pharmreg = result[['FID', 'name', 'nom', 'addr-postcode']]

In [ ]:
pharmreg.head(5)

,FID,name,nom,addr-postcode
0,pharmacies_point.4534591198,Pharmacie Centrale de Bondy,Île-de-France,93140
1,pharmacies_point.172156934,Pharmacie des Andaines,Normandie,NaN
2,pharmacies_point.3918927589,Pharmacie Cayeux Etaploise,Hauts-de-France,62630
3,pharmacies_point.5101170481,Pharmacie Lorraine,Grand Est,NaN
4,pharmacies_point.11429216164,Pharmacie de la Halle,Normandie,NaN


In [ ]:
grandest = pharmreg[pharmreg['nom'] == 'Grand Est'][['FID', 'name']]

In [ ]:
grandest.shape

(1471, 2)

Je retrouve 1471 pharmacies au grand-est contre 1368 pharmacies trouvées par Nedah

In [ ]:
cols = [
    "addr-housenumber",
    "addr-street",
    "addr-postcode",
    "addr-city"
]

result[cols].isna().sum()

,0
addr-housenumber,12112
addr-street,11924
addr-postcode,14758
addr-city,14988


In [ ]:
result[["name", "description"]].head(20)

,name,description
0,Pharmacie Centrale de Bondy,NaN
1,Pharmacie des Andaines,NaN
2,Pharmacie Cayeux Etaploise,NaN
3,Pharmacie Lorraine,NaN
4,Pharmacie de la Halle,NaN
5,Pharmacie Jeannot,NaN
6,Pharmacie du Square,NaN
7,Pharmacie Lambert,NaN
8,Centre de Santé Communal des Sources,NaN
9,Pharmacie Ravon,NaN


In [ ]:
result["description"].notna().sum()

np.int64(86)

In [ ]:
result["website"].notna().sum()
result["phone"].notna().sum()

np.int64(7648)

In [ ]:
print(len(result))

19137


In [ ]:
result[["name", "geometry"]].head()

,name,geometry
0,Pharmacie Centrale de Bondy,POINT (276175.853 6260215.63)
1,Pharmacie des Andaines,POINT (-40562.006 6204977.068)
2,Pharmacie Cayeux Etaploise,POINT (182438.63 6535708.512)
3,Pharmacie Lorraine,POINT (767638.826 6306264.503)
4,Pharmacie de la Halle,POINT (69996.839 6234936.367)


In [ ]:
print(result.crs)

EPSG:3857


Pour pouvoir retrouver les adresse sur maps il faut convertir nos coordonnées qui sont actuellement en 3857 Web Mercator, en format 4326 GPS standard de maps

conversion en coordonnées gps

In [ ]:
pharma_4326 = result.to_crs("EPSG:4326")

In [ ]:
pharma_4326[["name", "geometry"]].head(2)

,name,geometry
0,Pharmacie Centrale de Bondy,POINT (2.48093 48.91361)
1,Pharmacie des Andaines,POINT (-0.36437 48.58643)


In [ ]:
pharma_4326["longitude"] = pharma_4326.geometry.x
pharma_4326["latitude"] = pharma_4326.geometry.y

On va ensuite télécharger la BASE des Adresses Nationales sur le site https://api-adresse.data.gouv.fr/reverse/ pour faire du geocodage inversé afin de retrouver les adresses de chaque pharcie sur google maps

Nous allons choisir un petit échantillon pour tester l'API du BAN

In [ ]:
test = pharma_4326.copy()

In [ ]:
#test = pharma_4326.head(10).copy()

Testons le géocodage inverse du BAN

In [ ]:
import requests
#import pandas as pd

def reverse_ban(lat, lon):
    url = f"https://api-adresse.data.gouv.fr/reverse/?lon={lon}&lat={lat}"

    try:
        r = requests.get(url, timeout=10)
        data = r.json()

        if len(data["features"]) > 0:
            props = data["features"][0]["properties"]

            return pd.Series({
                "numero": props.get("housenumber"),
                "rue": props.get("street"),
                "code_postal": props.get("postcode"),
                "ville": props.get("city"),
                "adresse_complete": props.get("label")
            })

    except:
        pass

    return pd.Series({
        "numero": None,
        "rue": None,
        "code_postal": None,
        "ville": None,
        "adresse_complete": None
    })

In [ ]:
adresses = test.apply(
    lambda row: reverse_ban(row["latitude"], row["longitude"]),
    axis=1
)

test = pd.concat([test, adresses], axis=1)

In [ ]:
test[
    [
        "FID",
        "name",
        "adresse_complete",
        "ville",
        "code_postal"
    ]
]

,FID,name,adresse_complete,ville,code_postal
0,pharmacies_point.4534591198,Pharmacie Centrale de Bondy,26 Avenue Suzanne Buisson 93140 Bondy,Bondy,93140
1,pharmacies_point.172156934,Pharmacie des Andaines,53 Rue Félix Desaunay 61600 La Ferté Macé,La Ferté Macé,61600
2,pharmacies_point.3918927589,Pharmacie Cayeux Etaploise,32 Place du Général de Gaulle 62630 Étaples,Étaples,62630
3,pharmacies_point.5101170481,Pharmacie Lorraine,Rue Nationale 57600 Forbach,Forbach,57600
4,pharmacies_point.11429216164,Pharmacie de la Halle,19 Rue Saint-jean 61300 L'Aigle,L'Aigle,61300
...,...,...,...,...,...
19132,pharmacies_point.1008156074,Pharmacie Breuzin,Rue des Frères Voisin 37170 Chambray-lès-Tours,Chambray-lès-Tours,37170
19133,pharmacies_point.13179599050,Pharmacie de la République,1a Avenue de la Republique 59113 Seclin,Seclin,59113
19134,pharmacies_point.3712079795,Pharmacie des Arènes,1 Rue de Queuleu 57070 Metz,Metz,57070
19135,pharmacies_point.1933342224,Pharmacie Jacques,14 Rue Charles Gérome 88270 Dompaire,Dompaire,88270


In [ ]:
test["adresse_complete"].notna().mean()

np.float64(0.9959241260385641)

In [ ]:
test["adresse_complete"].isna().sum()

np.int64(78)

On va ensuite faire un test aléatoire sur 100 pharmacie

In [ ]:
#test100 = pharma_4326.sample(100, random_state=42)

In [ ]:
#adresses100 = test100.apply(
    #lambda row: reverse_ban(row["latitude"], row["longitude"]),
    #axis=1
#)

#test100 = pd.concat([test100, adresses100], axis=1)

#test100["adresse_complete"].notna().mean()

On va créer notre base finale et l'exporter

In [ ]:
pharm_final = test[
    [
        "FID",
        "name",
        "adresse_complete",
        "ville",
        "code_postal",
        "nom",
        "longitude",
        "latitude",
        "osm_id"
    ]
].copy()

In [ ]:
pharm_final.to_csv(
    "pharma_france.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
import os

os.listdir()

['.config',
 'pharmacies_point.csv',
 'pharma_france.csv',
 'regions.geojson',
 'sample_data']

In [ ]:
from google.colab import files

files.download("pharma_france.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Pour avancer dans notre travail nous avons décidé de nous répartir les différentes region pour le scraping sur google maps. Moi je vais scraper les regions suivantes :
Nouvelle aquitaine,
Occitanie,
Pays de la Loire,
Provence-Alpes-Côte d'Azur

In [ ]:
import pandas as pd
import geopandas as gpd
from shapely import wkt
import matplotlib.pyplot as plt

In [ ]:
pharm_final = pd.read_csv("/content/pharma_france-2.csv")

In [ ]:
pharm_final["nom"].unique()

array(['Île-de-France', 'Normandie', 'Hauts-de-France', 'Grand Est',
       'Auvergne-Rhône-Alpes', 'Pays de la Loire', 'Occitanie',
       'Centre-Val de Loire', "Provence-Alpes-Côte d'Azur",
       'Nouvelle-Aquitaine', 'Bourgogne-Franche-Comté', 'Bretagne', nan,
       'Corse'], dtype=object)

In [ ]:
regions_cibles = [
    "Nouvelle-Aquitaine",
    "Occitanie",
    "Pays de la Loire",
    "Provence-Alpes-Côte d'Azur"
]

pharma_cibles = pharm_final[
    pharm_final["nom"].isin(regions_cibles)
]

print(len(pharma_cibles))
pharma_cibles["nom"].value_counts()

6616


,count
nom,
Nouvelle-Aquitaine,1994
Occitanie,1903
Provence-Alpes-Côte d'Azur,1690
Pays de la Loire,1029


In [ ]:
pharm_final["name"].isna().sum()

np.int64(1265)

In [ ]:
pharm_final["name"].isna().mean() * 100

np.float64(6.610231488739092)

In [ ]:
pharma_scraping = pharma_cibles[
    pharma_cibles["name"].notna()
].copy()

In [ ]:
#pharma_scraping = pharma_cibles[
#    pharma_cibles["name"].notna()
#].copy()

In [ ]:
#print(len(pharma_scraping))

6112


Commençons le scraping

In [ ]:
#pharma_scraping["requete_google"] = (
 #   pharma_scraping["name"].str.strip()
  #  + ", "
   # + pharma_scraping["adresse_complete"].str.strip()
#)

Testons sur un petit échantillon

In [ ]:
#test50 = pharma_scraping.sample(
 #   50,
  #  random_state=42
#).copy()

In [ ]:
#test50[
 #   ["FID", "name", "adresse_complete", "nom"]
#].head()

,FID,name,adresse_complete,nom
9134,pharmacies_point.2052546659,Pharmacie Dethoor,78 Route de Vars 16430 Balzac,Nouvelle-Aquitaine
1030,pharmacies_point.12956181370,Pharmacie de Moux,43 Avenue Henri Bataille 11700 Moux,Occitanie
11491,pharmacies_point.3614697307,Pharmacie Principale,1 Rue Ambroise Croizat 65320 Bordères-sur-l'Échez,Occitanie
18458,pharmacies_point.414655734,Pharmacie de la Chasse Royale,87 Avenue Louis Cordelet 72000 Le Mans,Pays de la Loire
14590,pharmacies_point.2191008198,Pharmacie Labussiere,25 Route de Créon 33550 Langoiran,Nouvelle-Aquitaine


In [ ]:
!pip install playwright
!playwright install chromium

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.5/47.5 MB 19.0 MB/s eta 0:00:00
175.4 MiB [] 0% 361.4s175.4 MiB [] 0% 28.7s175.4 MiB [] 0% 18.0s175.4 MiB [] 0% 15.3s175.4 MiB [] 0% 9.2s175.4 MiB [] 1% 5.0s175.4 MiB [] 1% 5.7s175.4 MiB [] 2% 6.4s175.4 MiB [] 2% 6.6s175.4 MiB [] 2% 6.9s175.4 MiB [] 2% 7.2s175.4 MiB [] 2% 6.9s175.4 MiB [] 3% 5.7s175.4 MiB [] 4% 5.3s175.4 MiB [] 4% 5.1s175.4 MiB [] 5% 4.8s175.4 MiB [] 5% 4.5s175.4 MiB [] 6% 4.2s175.4 MiB [] 7% 4.3s175.4 MiB [] 7% 4.2s175.4 MiB [] 7% 4.3s175.4 MiB [] 8% 4.4s175.4 MiB [] 8% 4.5s175.4 MiB [] 8% 4.3s175.4 MiB [] 9% 4.0s175.4 MiB [] 10% 3.8s175.4 MiB [] 11% 3.5s175.4 MiB [] 12% 3.4s175.4 MiB [] 13% 3.3s175.4 MiB [] 14% 3.2s175.4 MiB [] 14% 3.1s175.4 MiB [] 14% 3.2s175.4 MiB [] 15% 3.1s175.4 MiB [] 16% 3.0s175.4 MiB [] 17% 2.8s175.4 MiB [] 18% 2.7s175.4 MiB [] 19% 2.6s175.4 MiB [] 20% 2.6s175.4 MiB [] 20% 2.7s175.4 MiB [] 20% 2.8s175.4 MiB [] 21% 2.7s175.4 MiB [] 22% 2.6s175.4 MiB [] 24% 2.4s175.4 MiB [] 25% 2.4s175.4 MiB [] 26% 

In [ ]:
pharmacie_test = test50.iloc[0]

requete = (
    pharmacie_test["name"]
    + ", "
    + pharmacie_test["adresse_complete"]
)

print(requete)

Pharmacie Dethoor, 78 Route de Vars 16430 Balzac


In [ ]:
from urllib.parse import quote

url = (
    "https://www.google.com/maps/search/"
    + quote(requete)
)

print(url)

https://www.google.com/maps/search/Pharmacie%20Dethoor%2C%2078%20Route%20de%20Vars%2016430%20Balzac


In [ ]:
# Install required system libraries for Playwright's Chromium
!apt-get update && apt-get install -y libatk-bridge2.0-0 libgtk-3-0 libgbm-dev

from playwright.async_api import async_playwright
import asyncio

async def run_playwright_script():
    async with async_playwright() as p:
        browser = await p.chromium.launch(
            headless=True # Changed to headless=True to run without a display server
        )

        page = await browser.new_page()

        await page.goto(url)

        await page.wait_for_timeout(5000)

        print(await page.title())

        await browser.close()

await run_playwright_script()

Hit:1 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:2 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:3 https://cli.github.com/packages stable InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
libatk-bridge2.0-0 is already the newest version (2.38.0-3).
libgbm-dev is already the newest 